# Task 3: train and evaluate meta-review models on Colab

Runtime → Change runtime type → **T4 GPU** (or L4/A100 if available). Then run the cells in order.

**Before you start:** upload your local GameWikiSum folder (`train.json`, `val.json`, `test.json`)
to Google Drive at `MyDrive/gamewikisum/` (or change `DATA_DRIVE` below).

Everything (models, metrics, predictions) is saved to `MyDrive/steam-review-translator-task3/`,
so a disconnect does not lose finished work. Re-running a cell skips a model or evaluation that
is already saved.

| System | What |
|---|---|
| B1 | extractive TextRank (k=3), no ML |
| B2 | `facebook/bart-large-cnn` zero-shot |
| E1 | BART-base fine-tuned, text only |
| E2 | BART-base fine-tuned + aspect prefix (main system) |
| E3 | T5-base fine-tuned + aspect prefix (optional) |

**Time budget (T4, rough):** each fine-tune at `PILOT=2000` takes about 30-45 min; each test
evaluation at `TEST_LIMIT=500` about 10-20 min. `PILOT=0` = full train split (hours per model).

In [9]:
# --- settings -------------------------------------------------------------------------
%cd /content
!rm -rf /content/srt
BRANCH = 'task3/local-gamewikisum'   # change to 'main' once that branch is merged
DATA_DRIVE = '/content/drive/MyDrive/dataset_nlp_tubes'
OUT = '/content/drive/MyDrive/dataset_nlp_tubes/out'
PILOT = 2000        # train documents per model (0 = full train split)
EPOCHS = 3
TEST_LIMIT = 500    # test documents to evaluate (None = all 1,454)
RUN_T5 = False      # E3 is optional; set True if there is time
SEED = 42

from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs(OUT, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

/content
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [10]:
# Clone the repo and install Task 3 dependencies.
!rm -rf /content/srt
!git clone -q --branch $BRANCH https://github.com/mahesa005/steam-review-translator.git /content/srt
%cd /content/srt
!pip install -q -e ".[task3]" 2>&1 | grep -v -i "warning" | tail -3

/content/srt


In [11]:
# Copy the data to the local disk (much faster to read than Drive).
!mkdir -p /content/gws && cp -v $DATA_DRIVE/train.json $DATA_DRIVE/val.json $DATA_DRIVE/test.json /content/gws/
!ls -la /content/gws
DATA = '/content/gws'

'/content/drive/MyDrive/dataset_nlp_tubes/train.json' -> '/content/gws/train.json'
'/content/drive/MyDrive/dataset_nlp_tubes/val.json' -> '/content/gws/val.json'
'/content/drive/MyDrive/dataset_nlp_tubes/test.json' -> '/content/gws/test.json'
total 2761516
drwxr-xr-x 2 root root       4096 Nov 22 19:59 .
drwxr-xr-x 1 root root       4096 Nov 22 20:07 ..
-rw------- 1 root root  324029474 Nov 22 20:08 test.json
-rw------- 1 root root 2183757555 Nov 22 20:08 train.json
-rw------- 1 root root  319986382 Nov 22 20:08 val.json


## 1. Dataset statistics (for the paper and slides)

In [14]:
import sys, json, statistics
from pathlib import Path
sys.path.insert(0, '/content/srt/experiments/task3_summarization')
from common import load_local_split

stats = {}
for split in ['train', 'val', 'test']:
    rows = load_local_split(Path(DATA), split)
    para = [len(r['preprocessed_paragraphs_per_reviews']) for r in rows]
    in_words = [sum(len(p.split()) for p in r['preprocessed_paragraphs_per_reviews']) for r in rows]
    sum_words = [len(r['preprocessed_summary'].split()) for r in rows]
    stats[split] = {
        'documents': len(rows),
        'games': len({r['game_title'] for r in rows}),
        'paragraphs_per_doc_mean (cap 400)': round(statistics.mean(para), 1),
        'input_words_mean (cap 400 paragraphs)': round(statistics.mean(in_words)),
        'summary_words_mean': round(statistics.mean(sum_words), 1),
        'summary_words_median': statistics.median(sum_words),
    }
    if split == 'train':
        example = rows[0]
    del rows
print(json.dumps(stats, indent=2))
json.dump(stats, open(f'{OUT}/dataset_stats.json', 'w'), indent=2)

print('\n=== training example ===')
print('game:', example['game_title'])
print('review paragraphs (first 3):')
for p in example['preprocessed_paragraphs_per_reviews'][:3]:
    print('  -', p[:300])
print('target summary:', example['preprocessed_summary'][:800])

2026-11-22 20:32:49,560 INFO steam_review_translator.common loaded train from /content/gws/train.json: 11744 documents
2026-11-22 20:33:26,435 INFO steam_review_translator.common loaded val from /content/gws/val.json: 1454 documents
2026-11-22 20:33:55,990 INFO steam_review_translator.common loaded test from /content/gws/test.json: 1454 documents


{
  "train": {
    "documents": 11744,
    "games": 7006,
    "paragraphs_per_doc_mean (cap 400)": 162.8,
    "input_words_mean (cap 400 paragraphs)": 10800,
    "summary_words_mean": 459.4,
    "summary_words_median": 318.0
  },
  "val": {
    "documents": 1454,
    "games": 880,
    "paragraphs_per_doc_mean (cap 400)": 184.3,
    "input_words_mean (cap 400 paragraphs)": 12208,
    "summary_words_mean": 431.4,
    "summary_words_median": 310.0
  },
  "test": {
    "documents": 1454,
    "games": 862,
    "paragraphs_per_doc_mean (cap 400)": 183.6,
    "input_words_mean (cap 400 paragraphs)": 12289,
    "summary_words_mean": 464.8,
    "summary_words_median": 314.0
  }
}

=== training example ===
game: Hybrid
review paragraphs (first 3):
  - hybrid takes place in a post - apocalyptic earth where two factions , the variants and the paladins , are battling it out all over the world over “dark matter” . as soon as you start the game you will be asked to choose a faction . both are identic

In [15]:
# The same example after preprocessing: the exact model input (aspect prefix + selected text).
from transformers import AutoTokenizer
from common import build_examples, normalize_rows
tok = AutoTokenizer.from_pretrained('facebook/bart-base')
rows = normalize_rows([example], 'preprocessed_paragraphs_per_reviews', 'preprocessed_summary',
                      'game_title')
ex, _ = build_examples(rows, tokenizer=tok, budget_tokens=800, max_target_tokens=256, top_k=5)
if ex:
    print('INPUT:', ex[0].input_text[:1200])
    print('\nTARGET:', ex[0].target[:600])
else:
    print('this document has < 100 words after cleaning; pick another one')

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

2026-11-22 20:34:16,352 INFO steam_review_translator.common built 1/1 valid examples (100.0%)


INPUT: aspects: multiplayer (9, neutral) ; gameplay (6, neutral) ; price (3, neutral) ; content (2, neutral) ; performance (1, neutral) </s> hybrid is a unique experiment in multiplayer cover - based shooting , and it succeeds gloriously . the action is frantic and non - stop , and it never gets old launching from one point of cover to the next , taking out several enemies before you even touch the ground . by focusing strictly on cover mechanics and removing all the fat , 5th cell has crafted one of the most interesting and hyperactive multiplayer games i’ve ever played . there is a constant sense of satisfaction in every single moment of hybrid , from the near constant stream of xp and unlockables , to the deluge of summoned drone attacks , to the metagame that ties everything together . multiplayer junkies , your drug has arrived .
it probably shouldn ' t be a surprise that if anyone could actually do something fresh in the online shooter space at this point , it would be the people

## 2. Fine-tune (each model is saved to Drive)

In [16]:
def train(script, out_dir, extra=''):
    if os.path.exists(f'{out_dir}/config.json'):
        print('already trained, skipping:', out_dir)
        return
    cmd = (f'python experiments/task3_summarization/{script} --dataset {DATA} --output {out_dir} '
           f'--pilot-samples {PILOT} --epochs {EPOCHS} --seed {SEED} --fp16 {extra}')
    print(cmd)
    get_ipython().system(cmd + ' 2>&1 | grep -E "eval_rouge|Error|error|disimpan"')

# E2 (main system): BART-base + aspect prefix
train('train_bart.py', f'{OUT}/models/e2_bart', '--aspect-prefix')

python experiments/task3_summarization/train_bart.py --dataset /content/gws --output /content/drive/MyDrive/dataset_nlp_tubes/out/models/e2_bart --pilot-samples 2000 --epochs 3 --seed 42 --fp16 --aspect-prefix
                                               {'eval_loss': '3.574', 'eval_rouge1': '0.3167', 'eval_rouge2': '0.0761', 'eval_rougeL': '0.1783', 'eval_rougeLsum': '0.2049', 'eval_runtime': '241', 'eval_samples_per_second': '0.83', 'eval_steps_per_second': '0.207', 'epoch': '1'}
                                               {'eval_loss': '3.492', 'eval_rouge1': '0.3207', 'eval_rouge2': '0.07443', 'eval_rougeL': '0.179', 'eval_rougeLsum': '0.205', 'eval_runtime': '242.6', 'eval_samples_per_second': '0.824', 'eval_steps_per_second': '0.206', 'epoch': '2'}
                                               {'eval_loss': '3.47', 'eval_rouge1': '0.3187', 'eval_rouge2': '0.07321', 'eval_rougeL': '0.1753', 'eval_rougeLsum': '0.2001', 'eval_runtime': '243.2', 'eval_samples_per_second': '0.82

In [17]:
# E1: BART-base, text only (shows whether the aspect prefix helps)
train('train_bart.py', f'{OUT}/models/e1_bart')

python experiments/task3_summarization/train_bart.py --dataset /content/gws --output /content/drive/MyDrive/dataset_nlp_tubes/out/models/e1_bart --pilot-samples 2000 --epochs 3 --seed 42 --fp16 
                                               {'eval_loss': '3.574', 'eval_rouge1': '0.311', 'eval_rouge2': '0.07177', 'eval_rougeL': '0.1782', 'eval_rougeLsum': '0.203', 'eval_runtime': '233.9', 'eval_samples_per_second': '0.855', 'eval_steps_per_second': '0.214', 'epoch': '1'}
                                               {'eval_loss': '3.497', 'eval_rouge1': '0.31', 'eval_rouge2': '0.07238', 'eval_rougeL': '0.177', 'eval_rougeLsum': '0.2012', 'eval_runtime': '232', 'eval_samples_per_second': '0.862', 'eval_steps_per_second': '0.216', 'epoch': '2'}
                                               {'eval_loss': '3.473', 'eval_rouge1': '0.3196', 'eval_rouge2': '0.07313', 'eval_rougeL': '0.178', 'eval_rougeLsum': '0.2031', 'eval_runtime': '233.5', 'eval_samples_per_second': '0.856', 'eval_steps_

In [18]:
# E3 (optional): T5-base + aspect prefix
if RUN_T5:
    train('train_t5.py', f'{OUT}/models/e3_t5')

## 3. Evaluate on test (each system once)

In [19]:
LIMIT = f'--limit {TEST_LIMIT}' if TEST_LIMIT else ''
R = f'{OUT}/runs'
os.makedirs(R, exist_ok=True)

def evaluate(name, model, extra=''):
    if os.path.exists(f'{R}/{name}.json'):
        print('already evaluated, skipping:', name)
        return
    cmd = (f'python experiments/task3_summarization/evaluate.py --dataset {DATA} --split test '
           f'--model {model} --name {name} {LIMIT} --out {R}/{name}.json '
           f'--records {R}/{name}_records.json {extra}')
    print(cmd)
    get_ipython().system(cmd + ' 2>&1 | grep -v -i warning | tail -25')

evaluate('B1', 'facebook/bart-base', '--extractive textrank --aspect-prefix')
evaluate('B2', 'facebook/bart-large-cnn')
evaluate('E1', f'{OUT}/models/e1_bart')
evaluate('E2', f'{OUT}/models/e2_bart', f'--aspect-prefix --baseline {R}/E1.json')   # E2 vs E1
if RUN_T5:
    evaluate('E3', f'{OUT}/models/e3_t5', f'--aspect-prefix --baseline {R}/E2.json')  # E3 vs E2

python experiments/task3_summarization/evaluate.py --dataset /content/gws --split test --model facebook/bart-base --name B1 --limit 500 --out /content/drive/MyDrive/dataset_nlp_tubes/out/runs/B1.json --records /content/drive/MyDrive/dataset_nlp_tubes/out/runs/B1_records.json --extractive textrank --aspect-prefix
2026-11-22 22:06:10,929 INFO steam_review_translator.common loaded test from /content/gws/test.json: 1454 documents
2026-11-22 22:06:49,975 INFO steam_review_translator.common built 499/500 valid examples (99.8%)
2026-11-22 22:06:49,976 INFO steam_review_translator.__main__ sampel valid: 499 dari 500
2026-11-22 22:07:02,191 INFO steam_review_translator.__main__ record per sampel ditulis ke /content/drive/MyDrive/dataset_nlp_tubes/out/runs/B1_records.json
2026-11-22 22:07:02,202 INFO steam_review_translator.__main__ hasil ditulis ke /content/drive/MyDrive/dataset_nlp_tubes/out/runs/B1.json
{
  "system": "B1",
  "model": "facebook/bart-base",
  "n_samples": 499,
  "valid_ratio": 

## 4. Results table

In [20]:
import glob
table = []
for f in sorted(glob.glob(f'{R}/*.json')):
    if f.endswith('_records.json') or f.endswith('error_analysis.json'):
        continue
    r = json.load(open(f))
    g = r['rouge']
    table.append((r['system'], r['n_samples'], g['rouge1'], g['rouge2'], g['rougeL'],
                  r['aspect_coverage'], r['polarity_consistency'],
                  r.get('bootstrap_vs_baseline', '')))
print(f"{'system':<6} {'n':>5} {'R-1':>6} {'R-2':>6} {'R-L':>6} {'asp.cov':>8} {'pol.cons':>9}")
for s, n, r1, r2, rl, cov, pol, bs in table:
    print(f'{s:<6} {n:>5} {r1:>6.3f} {r2:>6.3f} {rl:>6.3f} {cov:>8.3f} {pol:>9.3f}')
for s, *_, bs in table:
    if bs:
        print(f'{s} paired bootstrap vs baseline:', bs)

system     n    R-1    R-2    R-L  asp.cov  pol.cons
B1       499  0.177  0.028  0.109    0.291     0.596
B2       499  0.193  0.030  0.121    0.385     0.655
E1       499  0.311  0.077  0.180    0.340     0.954
E2       499  0.320  0.077  0.180    0.379     0.964
E2 paired bootstrap vs baseline: {'mean_diff': -0.00012540501879295153, 'p_value': 0.9080919080919081, 'ci_low': -0.0024169007328849225, 'ci_high': 0.002034722004844821}


## 5. Error analysis

In [21]:
systems = ' '.join(f'--system {s}={R}/{s}_records.json' for s in ['B2', 'E1', 'E2', 'E3']
                   if os.path.exists(f'{R}/{s}_records.json'))
get_ipython().system(f'python experiments/task3_summarization/error_analysis.py {systems} '
                     f'--out {R}/error_analysis.json 2>&1 | grep -v -i warning')

2026-11-22 22:40:05,472 INFO steam_review_translator.__main__ laporan ditulis ke /content/drive/MyDrive/dataset_nlp_tubes/out/runs/error_analysis.json

=== B2 (n=50) ===
  halusinasi           40
  repetisi             0
  aspek_terlewat       0
  gaya_tidak_sesuai    8
  kegagalan_input      0
  tidak_ada_error      2

=== E1 (n=50) ===
  halusinasi           0
  repetisi             0
  aspek_terlewat       2
  gaya_tidak_sesuai    45
  kegagalan_input      0
  tidak_ada_error      3

=== E2 (n=50) ===
  halusinasi           1
  repetisi             0
  aspek_terlewat       0
  gaya_tidak_sesuai    48
  kegagalan_input      0
  tidak_ada_error      1


In [22]:
# Example outputs for the slides: worst, median and best E2 sample by ROUGE-L.
recs = json.load(open(f'{R}/E2_records.json'))
order = sorted(range(len(recs)), key=lambda i: recs[i]['rougeL'])
for label, i in [('worst', order[0]), ('median', order[len(order) // 2]), ('best', order[-1])]:
    r = recs[i]
    print(f"=== {label}: {r['game']} (ROUGE-L {r['rougeL']:.3f}) ===")
    print('aspects  :', r['aspects'])
    print('predicted:', r['prediction'][:700])
    print('reference:', r['reference'][:700], '\n')

=== worst: Anito: Defend a Land Enraged (ROUGE-L 0.047) ===
aspects  : ['content', 'multiplayer', 'story']
predicted: super mario bros is an action role - playing game played from a third - person perspective . the player controls the main character in the game ' s story . the game is set in a fantasy world , with the player controlling the main characters in the story . players can choose from a variety of characters , such as a wolf , a fox , and a wolf . the main protagonist is a wolf named wolf , who can be controlled by the player . wolf can also be controlled through the player ' s character ' s actions , and wolf can use his powers to control the character . wolf is able to use his abilities to attack enemies , and he can also use his special powers to attack them .
reference: players begin with choosing between two characters : maya and agila . 

=== median: Age of Empires: Castle Siege (ROUGE-L 0.182) ===
aspects  : ['audio', 'gameplay', 'graphics']
predicted: the gameplay of 

## 6. Send back

Copy the printed results table, the error-analysis table and the example outputs into
`experiments/task3_summarization/RESULTS.md` (or the group chat). Everything is also in
`MyDrive/steam-review-translator-task3/` (`dataset_stats.json`, `runs/*.json`, `models/`).
Do **not** commit model weights or GameWikiSum data to git.